In [11]:
# ACA 2 · Ingeniería de características, pruebas e indicadores
# Datos: base limpia del ACA 1 (repositorio GitHub, versión v0.1)
import pandas as pd
import numpy as np

SEMILLA = 42                     # semilla fija para que todo sea reproducible
np.random.seed(SEMILLA)

URL = ("https://raw.githubusercontent.com/JAIRO-ORREGO/desercion-academica-pregrado/"
       "v0.1/datos/desercion_academica_pregrado_limpio.csv")
df = pd.read_csv(URL)

print("Filas y columnas:", df.shape)
df.head()

Filas y columnas: (3091, 16)


,periodo_cod,anio,semestre,nombre_facultad,nombre_programa,nombre_sede,modalidad,jornada,genero,estrato,edad,rango_edad,origen_geografico,norma,literal,causal
0,2020-1,2020,1,SECCIONAL TUNJA,TECNICO PROFESIONAL EN INSTALACION Y MANTENIMIENTO DE RE...,TUNJA,VIRTUAL,DIURNA - NOCTURNA,F,SIN INFORMACIÓN,19,16-20,PAIPA,ACUERDO 130 DE 1998 (ART. 80),B,130-B
1,2020-1,2020,1,SECCIONAL TUNJA,TECNICO PROFESIONAL EN INSTALACION Y MANTENIMIENTO DE RE...,TUNJA,VIRTUAL,DIURNA - NOCTURNA,F,SIN INFORMACIÓN,17,16-20,PAIPA,ACUERDO 130 DE 1998 (ART. 80),B,130-B
2,2020-1,2020,1,SECCIONAL CHIQUINQUIRA,TECNICA PROFESIONAL EN PROCESOS AGROINDUSTRIALES,CHIQUINQUIRA,VIRTUAL,DIURNA - NOCTURNA,M,SIN INFORMACIÓN,16,16-20,CALDAS,ACUERDO 130 DE 1998 (ART. 80),B,130-B
3,2020-1,2020,1,SECCIONAL CHIQUINQUIRA,TECNICA PROFESIONAL EN PROCESOS AGROINDUSTRIALES,CHIQUINQUIRA,VIRTUAL,DIURNA - NOCTURNA,M,SIN INFORMACIÓN,17,16-20,CALDAS,ACUERDO 130 DE 1998 (ART. 80),B,130-B
4,2021-1,2021,1,CIENCIAS,FISICA,TUNJA,PRESENCIAL,DIURNA,F,2,23,21-25,BOGOTÁ D.C.,ACUERDO 130 DE 1998 (ART. 80),B,130-B


In [12]:
import unicodedata

def sin_tildes(texto):
    """Quita tildes y espacios extra para poder comparar nombres de municipios."""
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return texto.upper().strip()

# 1. Sede agrupada: las 4 sedes con más registros y el resto en «OTRAS SEDES»
principales = ["TUNJA", "SOGAMOSO", "DUITAMA", "CHIQUINQUIRA"]
df["sede_grupo"] = df["nombre_sede"].where(df["nombre_sede"].isin(principales), "OTRAS SEDES")

# 2. Facultad agrupada: las facultades con menos de 10 registros pasan a «OTRAS»
conteo_fac = df["nombre_facultad"].value_counts()
pequenas = conteo_fac[conteo_fac < 10].index
df["facultad_grupo"] = df["nombre_facultad"].where(~df["nombre_facultad"].isin(pequenas), "OTRAS")

# 3. Estrato agrupado: el 3 y el 4 se unen en «3 O MÁS»
df["estrato_grupo"] = df["estrato"].replace({"3": "3 O MÁS", "4": "3 O MÁS"})

# 4. Jornada agrupada: «DIURNA - NOCTURNA» (4 registros) pasa a «OTRA»
df["jornada_grupo"] = df["jornada"].replace({"DIURNA - NOCTURNA": "OTRA"})

# 5. Estudiante local: 1 si el municipio de origen es el mismo de la sede
origen = df["origen_geografico"].map(sin_tildes).replace({"BOGOTA D.C.": "BOGOTA"})
sede = df["nombre_sede"].map(sin_tildes).replace({"SANTA FE DE BOGOTA": "BOGOTA"})
df["es_local"] = (origen == sede).astype(int)

# 6. Orden del periodo: 1, 2, 3… según el orden cronológico
periodos = sorted(df["periodo_cod"].unique())
df["periodo_orden"] = df["periodo_cod"].map({p: i + 1 for i, p in enumerate(periodos)})

# Verificación de cada variable nueva
for col in ["sede_grupo", "facultad_grupo", "estrato_grupo", "jornada_grupo", "es_local", "periodo_orden"]:
    print(f"\n=== {col} ===")
    print(df[col].value_counts().sort_index())
print("\nFilas y columnas:", df.shape)


=== sede_grupo ===
sede_grupo
CHIQUINQUIRA     176
DUITAMA          406
OTRAS SEDES      385
SOGAMOSO         426
TUNJA           1698
Name: count, dtype: int64

=== facultad_grupo ===
facultad_grupo
CIENCIAS                                  178
CIENCIAS AGROPECUARIAS                     80
CIENCIAS DE LA EDUCACION                  526
CIENCIAS DE LA SALUD                       31
CIENCIAS ECONOMICAS Y ADMINISTRATIVAS     186
DERECHO Y CIENCIAS SOCIALES                43
ESTUDIOS A DISTANCIA                     1136
INGENIERIA                                293
OTRAS                                       3
SECCIONAL CHIQUINQUIRA                     80
SECCIONAL DUITAMA                         271
SECCIONAL SOGAMOSO                        264
Name: count, dtype: int64

=== estrato_grupo ===
estrato_grupo
1                   797
2                  1840
3 O MÁS             397
SIN INFORMACIÓN      57
Name: count, dtype: int64

=== jornada_grupo ===
jornada_grupo
DIURNA       1410
EXTENDI

In [13]:
import os
os.makedirs("results/tables", exist_ok=True)

variables_creadas = pd.DataFrame([
    ["sede_grupo", "Tunja, Sogamoso, Duitama y Chiquinquirá; el resto = «OTRAS SEDES»",
     "Las 15 sedes restantes tienen entre 1 y 75 registros; agruparlas asegura frecuencias suficientes para la prueba chi-cuadrado"],
    ["facultad_grupo", "Facultades con menos de 10 registros = «OTRAS»",
     "Seccional Tunja (2) y Sede Regional Aguazul (1) no permiten comparaciones válidas"],
    ["estrato_grupo", "Estratos 3 y 4 = «3 O MÁS»",
     "El estrato 4 tiene solo 19 registros"],
    ["jornada_grupo", "«DIURNA - NOCTURNA» = «OTRA»",
     "La categoría tiene solo 4 registros"],
    ["es_local", "1 si municipio de origen = municipio de la sede; 0 en otro caso",
     "Variable del dominio: aproxima si el estudiante se separó de su comunidad de origen, etapa que Tinto (1993) asocia con la integración y la permanencia"],
    ["periodo_orden", "Número consecutivo del periodo académico (1 = 2020-1 … 9 = 2025-1)",
     "Permite ordenar los periodos y calcular la variación entre periodos (KPI de la Tabla 1)"],
], columns=["variable", "fórmula", "justificación"])

variables_creadas.to_csv("results/tables/variables_creadas.csv", index=False, encoding="utf-8-sig")
variables_creadas

,variable,fórmula,justificación
0,sede_grupo,"Tunja, Sogamoso, Duitama y Chiquinquirá; el resto = «OTR...",Las 15 sedes restantes tienen entre 1 y 75 registros; ag...
1,facultad_grupo,Facultades con menos de 10 registros = «OTRAS»,Seccional Tunja (2) y Sede Regional Aguazul (1) no permi...
2,estrato_grupo,Estratos 3 y 4 = «3 O MÁS»,El estrato 4 tiene solo 19 registros
3,jornada_grupo,«DIURNA - NOCTURNA» = «OTRA»,La categoría tiene solo 4 registros
4,es_local,1 si municipio de origen = municipio de la sede; 0 en ot...,Variable del dominio: aproxima si el estudiante se separ...
5,periodo_orden,Número consecutivo del periodo académico (1 = 2020-1 … 9...,Permite ordenar los periodos y calcular la variación ent...


In [14]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif

# Variables candidatas y variable de referencia (la causal de pérdida de cupo)
candidatas = ["facultad_grupo", "sede_grupo", "modalidad", "jornada_grupo", "genero",
              "estrato_grupo", "rango_edad", "es_local", "periodo_cod"]
y = df["causal"]

# 1. Codificación: cada categoría se convierte en un número (ordinal encoding)
codificador = OrdinalEncoder()
X = pd.DataFrame(codificador.fit_transform(df[candidatas].astype(str)), columns=candidatas)

# Referencia: con codificación one-hot (una columna 0/1 por categoría) serían estas columnas
print("Columnas con one-hot:", pd.get_dummies(df[candidatas].astype(str)).shape[1])

# 2. Selección: información mutua de cada variable con la causal
im = mutual_info_classif(X, y, discrete_features=True, random_state=SEMILLA)
seleccion = (pd.DataFrame({"variable": candidatas, "informacion_mutua": im.round(4)})
             .sort_values("informacion_mutua", ascending=False))
UMBRAL = 0.01
seleccion["seleccionada"] = np.where(seleccion["informacion_mutua"] >= UMBRAL, "Sí", "No")

seleccion.to_csv("results/tables/seleccion_informacion_mutua.csv", index=False, encoding="utf-8-sig")
seleccion

Columnas con one-hot: 46


,variable,informacion_mutua,seleccionada
6,rango_edad,0.0465,Sí
0,facultad_grupo,0.0304,Sí
8,periodo_cod,0.0236,Sí
1,sede_grupo,0.0182,Sí
2,modalidad,0.0168,Sí
3,jornada_grupo,0.0166,Sí
5,estrato_grupo,0.0091,No
4,genero,0.0016,No
7,es_local,0.0005,No


In [15]:
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/desercion_academica_aca2.csv", index=False, encoding="utf-8-sig")
print("Guardado:", df.shape)

Guardado: (3091, 22)


In [16]:
# 6. Pruebas de hipótesis (objetivo específico 2): chi-cuadrado y V de Cramér con IC 95 % (bootstrap)
from scipy.stats import chisquare, chi2_contingency

B = 2000                                   # réplicas bootstrap
rng = np.random.default_rng(SEMILLA)

def v_bondad(obs):                         # V de Cramér para bondad de ajuste
    k, n = len(obs), obs.sum()
    return np.sqrt(chisquare(obs).statistic / (n * (k - 1)))

def v_indep(tab):                          # V de Cramér para independencia
    chi2 = chi2_contingency(tab, correction=False)[0]
    return np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1)))

def ic_bootstrap(d, f):
    vals = [f(d.sample(len(d), replace=True, random_state=rng.integers(1e9)).reset_index(drop=True))
            for _ in range(B)]
    return np.percentile(vals, [2.5, 97.5])

filas = []

# H1: la deserción se distribuye por igual entre los periodos (sin 2020-1, con 4 registros)
d1 = df[df["periodo_cod"] != "2020-1"]
obs = d1["periodo_cod"].value_counts().sort_index()
r = chisquare(obs)
ic = ic_bootstrap(d1, lambda s: v_bondad(s["periodo_cod"].value_counts().reindex(obs.index, fill_value=0).values))
filas.append(["H1", "Bondad de ajuste", "periodo_cod", len(d1), r.statistic, len(obs) - 1, r.pvalue, v_bondad(obs.values), *ic, 0])

# H2: causal independiente de la facultad (sin "OTRAS" y sin la causal 130-A, por frecuencias esperadas < 5)
# H3: modalidad independiente del periodo (sin 2020-1)
pruebas = [("H2", "facultad_grupo", "causal", df[(df["facultad_grupo"] != "OTRAS") & (df["causal"] != "130-A")]),
           ("H3", "periodo_cod", "modalidad", df[df["periodo_cod"] != "2020-1"])]
for h, a, b, d in pruebas:
    tab = pd.crosstab(d[a], d[b])
    chi2, p, gl, esp = chi2_contingency(tab, correction=False)
    ic = ic_bootstrap(d, lambda s: v_indep(pd.crosstab(s[a], s[b])))
    filas.append([h, "Independencia", f"{a} × {b}", len(d), chi2, gl, p, v_indep(tab), *ic, int((esp < 5).sum())])

pruebas_hipotesis = pd.DataFrame(filas, columns=["hipotesis", "prueba", "variables", "n", "chi2", "gl",
                                                 "valor_p", "v_cramer", "ic95_inf", "ic95_sup", "celdas_esperado_menor_5"])
pruebas_hipotesis["decision"] = np.where(pruebas_hipotesis["valor_p"] < 0.05, "Se rechaza H0", "No se rechaza H0")
pruebas_hipotesis.to_csv("pruebas_hipotesis.csv", index=False)
pruebas_hipotesis.round(4)

,hipotesis,prueba,variables,n,chi2,gl,valor_p,v_cramer,ic95_inf,ic95_sup,celdas_esperado_menor_5,decision
0,H1,Bondad de ajuste,periodo_cod,3087,178.7337,7,0.0,0.0909,0.0771,0.1072,0,Se rechaza H0
1,H2,Independencia,facultad_grupo × causal,3080,171.6570,30,0.0,0.1363,0.1268,0.1665,0,Se rechaza H0
2,H3,Independencia,periodo_cod × modalidad,3087,94.8079,14,0.0,0.1239,0.1079,0.1581,0,Se rechaza H0


In [17]:
# 7. Objetivo específico 3: evidencia de las limitaciones de calidad de la base original
URL_API = "https://www.datos.gov.co/resource/3iew-7wpx.csv?$limit=10000"
crudo = pd.read_csv(URL_API, dtype=str)

pct = lambda x: f"{x:.1%}".replace(".", ",").replace("%", " %")
posgrado = crudo["nombre_programa"].str.contains("ESPECIALIZACI|MAESTR|DOCTORADO", case=False)
periodo_irregular = ~crudo["periodo"].str.fullmatch(r"(PRIMER|SEGUNDO) SEMESTRE DE \d{4}")
doble_espacio = crudo[["periodo", "nombre_facultad", "nombre_programa"]].apply(lambda s: s.str.contains("  ")).any(axis=1)
programas_dup = crudo["nombre_programa"].nunique() - crudo["nombre_programa"].str.split().str.join(" ").nunique()
modalidad_may = crudo["modalidad"].str.isupper()
estrato_sin = crudo["estrato"].isna() | crudo["estrato"].str.upper().str.startswith("SIN INFORMACI", na=False)
periodos = df["periodo_cod"].value_counts().sort_index()
sin_registros = [f"{a}-{s}" for a in range(2020, 2026) for s in (1, 2)
                 if f"{a}-{s}" not in periodos.index and f"{a}-{s}" <= periodos.index.max()]

lim = [
    ["Sin columna de nivel de formación",
     f"{posgrado.sum()} registros de posgrado identificados solo por el nombre del programa",
     "El filtro de pregrado depende de palabras del nombre del programa",
     "Columna de nivel de formación (pregrado o posgrado)"],
    ["Periodo académico registrado como texto libre",
     f"{periodo_irregular.sum()} registros con etiquetas fuera del formato común "
     f"({', '.join(crudo.loc[periodo_irregular, 'periodo'].unique())})",
     "Fue necesario recodificar el periodo antes de cualquier comparación",
     "Código de periodo estandarizado (por ejemplo, 2021-1)"],
    ["Mayúsculas y espacios dobles en categorías",
     f"{modalidad_may.sum()} registros con la modalidad en mayúsculas ({crudo['modalidad'].nunique()} valores para "
     f"{crudo['modalidad'].str.upper().nunique()} categorías); {doble_espacio.sum()} registros con espacios dobles; "
     f"{programas_dup} programas duplicados por espacios",
     "Sin normalizar, una misma categoría se contaría como dos",
     "Listas de valores cerradas en el sistema de registro"],
    ["Error de digitación en la jornada",
     f"{(crudo['jornada'] == 'DIURNA - NOCTURA').sum()} registros con “DIURNA - NOCTURA”",
     "Categoría mal escrita que debió corregirse",
     "Listas de valores cerradas en el sistema de registro"],
    ["Estrato sin información",
     f"{estrato_sin.sum()} registros ({pct(estrato_sin.mean())}) vacíos o “SIN INFORMACIÓN”, escritos de "
     f"{crudo.loc[estrato_sin, 'estrato'].fillna('vacío').nunique()} formas",
     "Reduce la base útil para analizar el estrato",
     "Estrato obligatorio al momento de la matrícula"],
    ["Cobertura desigual de los periodos (pregrado)",
     "; ".join(f"{p}: {n}" for p, n in periodos.items()) + "; sin registros: " + ", ".join(sin_registros),
     "Los periodos no son comparables entre sí; 2020-1 se excluyó de las pruebas",
     "Registro completo de todos los periodos, o nota que explique los vacíos"],
    ["Causal registrada solo como norma, artículo y literal",
     f"{crudo['nombre_estado'].nunique()} causales sin descripción del motivo; la 130-A tiene "
     f"{(df['causal'] == '130-A').sum()} registros",
     "No se sabe qué situación académica describe cada causal; la 130-A se excluyó de la prueba H2",
     "Descripción de cada causal y motivo de la pérdida de cupo"],
    ["Sin población de referencia",
     "La base solo contiene estudiantes que perdieron el cupo",
     "Solo se pueden comparar conteos y proporciones internas, no tasas de deserción",
     "Número de matriculados por periodo, facultad, sede y modalidad"],
    ["Datos personales indirectos",
     f"Fecha de nacimiento exacta en {crudo['fecha_nacimiento'].notna().sum()} registros y lugar de expedición del documento",
     "Riesgo de reidentificación; la fecha se transformó en edad y el lugar de expedición no se usó",
     "Publicar la edad o el rango de edad en lugar de la fecha exacta"],
]
limitaciones = pd.DataFrame(lim, columns=["limitacion", "evidencia", "efecto_en_el_analisis", "dato_requerido"])
limitaciones.to_csv("limitaciones_calidad.csv", index=False)
pd.set_option("display.max_colwidth", None)
limitaciones

,limitacion,evidencia,efecto_en_el_analisis,dato_requerido
0,Sin columna de nivel de formación,281 registros de posgrado identificados solo por el nombre del programa,El filtro de pregrado depende de palabras del nombre del programa,Columna de nivel de formación (pregrado o posgrado)
1,Periodo académico registrado como texto libre,"659 registros con etiquetas fuera del formato común (PRIMER SEMESTRE 2020, PRIMER SEMESTRE DE 2021)",Fue necesario recodificar el periodo antes de cualquier comparación,"Código de periodo estandarizado (por ejemplo, 2021-1)"
2,Mayúsculas y espacios dobles en categorías,45 registros con la modalidad en mayúsculas (5 valores para 3 categorías); 814 registros con espacios dobles; 3 programas duplicados por espacios,"Sin normalizar, una misma categoría se contaría como dos",Listas de valores cerradas en el sistema de registro
3,Error de digitación en la jornada,45 registros con “DIURNA - NOCTURA”,Categoría mal escrita que debió corregirse,Listas de valores cerradas en el sistema de registro
4,Estrato sin información,"338 registros (10,0 %) vacíos o “SIN INFORMACIÓN”, escritos de 3 formas",Reduce la base útil para analizar el estrato,Estrato obligatorio al momento de la matrícula
5,Cobertura desigual de los periodos (pregrado),"2020-1: 4; 2021-1: 606; 2022-1: 344; 2022-2: 352; 2023-1: 278; 2023-2: 312; 2024-1: 404; 2024-2: 397; 2025-1: 394; sin registros: 2020-2, 2021-2",Los periodos no son comparables entre sí; 2020-1 se excluyó de las pruebas,"Registro completo de todos los periodos, o nota que explique los vacíos"
6,"Causal registrada solo como norma, artículo y literal",8 causales sin descripción del motivo; la 130-A tiene 8 registros,No se sabe qué situación académica describe cada causal; la 130-A se excluyó de la prueba H2,Descripción de cada causal y motivo de la pérdida de cupo
7,Sin población de referencia,La base solo contiene estudiantes que perdieron el cupo,"Solo se pueden comparar conteos y proporciones internas, no tasas de deserción","Número de matriculados por periodo, facultad, sede y modalidad"
8,Datos personales indirectos,Fecha de nacimiento exacta en 3372 registros y lugar de expedición del documento,Riesgo de reidentificación; la fecha se transformó en edad y el lugar de expedición no se usó,Publicar la edad o el rango de edad en lugar de la fecha exacta


In [18]:
# 8. Modelo dimensional (esquema estrella), KPI de la Tabla 1 y validación de totales
import os, shutil
os.makedirs("modelo_estrella", exist_ok=True)

def dimension(cols, clave):
    dim = df[cols].drop_duplicates().sort_values(cols).reset_index(drop=True)
    dim.insert(0, clave, range(1, len(dim) + 1))
    return dim

dims = {
    "dim_periodo":   dimension(["periodo_cod", "anio", "semestre", "periodo_orden"], "id_periodo"),
    "dim_programa":  dimension(["nombre_programa", "nombre_facultad", "facultad_grupo"], "id_programa"),
    "dim_sede":      dimension(["nombre_sede", "sede_grupo"], "id_sede"),
    "dim_oferta":    dimension(["modalidad", "jornada", "jornada_grupo"], "id_oferta"),
    "dim_causal":    dimension(["causal", "norma", "literal"], "id_causal"),
    "dim_perfil":    dimension(["genero", "estrato", "estrato_grupo", "rango_edad", "es_local"], "id_perfil"),
}
hechos = df.copy()
for nombre, dim in dims.items():
    hechos = hechos.merge(dim, on=list(dim.columns[1:]), how="left")
hechos.insert(0, "id_registro", range(1, len(hechos) + 1))
hechos = hechos[["id_registro"] + [d.columns[0] for d in dims.values()] + ["edad"]]
hechos["n_registros"] = 1

for nombre, dim in dims.items():
    dim.to_csv(f"modelo_estrella/{nombre}.csv", index=False)
hechos.to_csv("modelo_estrella/hechos_perdida_cupo.csv", index=False)

# ---- KPI de la Tabla 1 ----
N = len(df)
pct = lambda x: round(100 * x, 1)
kpi = []
for var in ["modalidad", "sede_grupo", "facultad_grupo"]:
    for cat, n in df[var].value_counts().items():
        kpi.append(["Participación por categoría (%)", var, cat, n, pct(n / N)])
por_periodo = df["periodo_cod"].value_counts().sort_index()
def anterior(p):                                  # semestre inmediatamente anterior: 2023-1 -> 2022-2
    a, s = map(int, p.split("-"))
    return f"{a}-1" if s == 2 else f"{a-1}-2"
for p, n in por_periodo.items():                  # solo entre semestres consecutivos con registros
    previo = anterior(p)
    valor = round((n - por_periodo[previo]) / por_periodo[previo] * 100, 1) if previo in por_periodo.index and previo != "2020-1" else None
    kpi.append(["Variación entre periodos (%)", "periodo_cod", p, n, valor])
top3 = df["nombre_facultad"].value_counts().head(3)
kpi.append(["Concentración en las tres facultades con más registros (%)", "nombre_facultad",
            " + ".join(top3.index), top3.sum(), pct(top3.sum() / N)])
for cat, n in df["causal"].value_counts().items():
    kpi.append(["Participación por causal (%)", "causal", cat, n, pct(n / N)])
pregrado_crudo = crudo[~crudo["nombre_programa"].str.contains("ESPECIALIZACI|MAESTR|DOCTORADO", case=False)]
estrato_sin = pregrado_crudo["estrato"].isna() | pregrado_crudo["estrato"].str.upper().str.startswith("SIN INFORMACI", na=False)
calidad = {
    "estrato (vacío o “SIN INFORMACIÓN”)": estrato_sin.sum(),
    "modalidad (escrita en mayúsculas)": pregrado_crudo["modalidad"].str.isupper().sum(),
    "jornada (“DIURNA - NOCTURA”)": (pregrado_crudo["jornada"] == "DIURNA - NOCTURA").sum(),
    "periodo (fuera del formato común)": (~pregrado_crudo["periodo"].str.fullmatch(r"(PRIMER|SEGUNDO) SEMESTRE DE \d{4}")).sum(),
}
for var, n in calidad.items():
    kpi.append(["Calidad del dato (%)", var, "registros inconsistentes o genéricos", n, pct(n / len(pregrado_crudo))])
kpi_resultados = pd.DataFrame(kpi, columns=["kpi", "variable", "categoria", "registros", "valor_pct"])
kpi_resultados.to_csv("kpi_resultados.csv", index=False)

# ---- Validación de totales: fuente (API) vs. modelo ----
validacion = pd.DataFrame([
    ["Registros de pregrado en la fuente (API, sin posgrado)", len(pregrado_crudo)],
    ["Registros en la tabla de hechos", len(hechos)],
    ["Suma de n_registros en la tabla de hechos", hechos["n_registros"].sum()],
    ["Registros de hechos sin dimensión asociada", int(hechos.isna().any(axis=1).sum())],
] + [[f"Suma de registros por {d}", hechos.groupby(dims[d].columns[0])["n_registros"].sum().sum()] for d in dims],
    columns=["control", "valor"])
validacion["diferencia_con_fuente"] = validacion["valor"] - len(pregrado_crudo)
validacion.loc[3, "diferencia_con_fuente"] = None
validacion.to_csv("validacion_totales.csv", index=False)

shutil.make_archive("modelo_estrella", "zip", "modelo_estrella")
print({k: v.shape for k, v in dims.items()}, hechos.shape)
print(validacion.to_string())
print(kpi_resultados.to_string())

{'dim_periodo': (9, 5), 'dim_programa': (114, 4), 'dim_sede': (19, 3), 'dim_oferta': (10, 4), 'dim_causal': (5, 4), 'dim_perfil': (83, 6)} (3091, 9)
                                                  control  valor  diferencia_con_fuente
0  Registros de pregrado en la fuente (API, sin posgrado)   3091                    0.0
1                         Registros en la tabla de hechos   3091                    0.0
2               Suma de n_registros en la tabla de hechos   3091                    0.0
3              Registros de hechos sin dimensión asociada      0                    NaN
4                       Suma de registros por dim_periodo   3091                    0.0
5                      Suma de registros por dim_programa   3091                    0.0
6                          Suma de registros por dim_sede   3091                    0.0
7                        Suma de registros por dim_oferta   3091                    0.0
8                        Suma de registros por dim_causal  

In [19]:
# 9. KPI por programa académico y comparación de los KPI con su referencia (Tabla 1)
N = len(df)
pct = lambda x: round(100 * x, 1)

# a) Programas: los 10 con más registros y su participación
prog = df.groupby(["nombre_programa", "nombre_facultad"]).size().sort_values(ascending=False).rename("registros").reset_index()
prog["participacion_pct"] = (prog["registros"] / N * 100).round(1)
top10 = prog.head(10).copy()
por_programa = prog["registros"]                     # programa = nombre + facultad (114 combinaciones, como en dim_programa)
resumen_programas = pd.DataFrame({
    "indicador": ["Programas con registros", "Programas con menos de 5 registros", "Participación de los 10 programas principales (%)"],
    "valor": [por_programa.size, int((por_programa < 5).sum()), pct(top10["registros"].sum() / N)]})
top10.to_csv("kpi_top10_programas.csv", index=False)

# b) Referencia de cada KPI: reparto uniforme entre las categorías (Tabla 1)
filas = []
for var, nombre in [("modalidad", "Modalidad"), ("sede_grupo", "Sede"), ("facultad_grupo", "Facultad")]:
    k = df[var].nunique()
    for cat, n in df[var].value_counts().head(3).items():
        filas.append([f"Participación por categoría – {nombre}", cat, pct(n / N), pct(1 / k)])
k_fac = df["nombre_facultad"].nunique()
top3 = df["nombre_facultad"].value_counts().head(3)
filas.append(["Concentración en las tres facultades con más registros", " + ".join(top3.index), pct(top3.sum() / N), pct(3 / k_fac)])
k_prog = por_programa.size
filas.append(["Concentración en los 10 programas con más registros", "10 programas", pct(top10["registros"].sum() / N), pct(10 / k_prog)])
kpi_vs_ref = pd.DataFrame(filas, columns=["kpi", "categoria", "valor_pct", "referencia_uniforme_pct"])
kpi_vs_ref["veces_la_referencia"] = (kpi_vs_ref["valor_pct"] / kpi_vs_ref["referencia_uniforme_pct"]).round(1)
kpi_vs_ref.to_csv("kpi_referencias.csv", index=False)

pd.set_option("display.max_colwidth", 60)
print(resumen_programas.to_string(index=False), "\n")
print(top10.to_string(index=False), "\n")
print(kpi_vs_ref.to_string(index=False))

                                        indicador  valor
                          Programas con registros  114.0
               Programas con menos de 5 registros   21.0
Participación de los 10 programas principales (%)   38.6 

                                          nombre_programa                       nombre_facultad  registros  participacion_pct
TECNICA PROFESIONAL EN PROCESOS COMERCIALES Y FINANCIEROS                  ESTUDIOS A DISTANCIA        305                9.9
 TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS DE SALUD                  ESTUDIOS A DISTANCIA        218                7.1
                              TECNOLOGIA EN OBRAS CIVILES                  ESTUDIOS A DISTANCIA        194                6.3
                              LICENCIATURA EN MATEMATICAS              CIENCIAS DE LA EDUCACION         86                2.8
                                                 ECONOMIA CIENCIAS ECONOMICAS Y ADMINISTRATIVAS         80                2.6
              

In [20]:
# 10. Entorno de cómputo (para el README)
import sys, platform
print("Python:", sys.version.split()[0], "| Sistema:", platform.platform())
!pip freeze > requirements.txt

Python: 3.13.18 | Sistema: Linux-6.6.159+-x86_64-with-glibc2.39
